# Задание 9. Тематическое моделирование с использованием латентного размещения Дирихле

Входной текст: `lab02_data.txt` (10 предложений, по условию содержит две темы). Модель: LDA из `gensim`, `num_topics = 2`.

In [1]:
import nltk
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
from gensim import models, corpora

nltk.download("stopwords", quiet=True)

[nltk_data] Error loading stopwords: Security Violation
[nltk_data]     [pathsec.urlopen]: refusing a proxied fetch of
[nltk_data]     'https://raw.githubusercontent.com/nltk/nltk_data/gh-
[nltk_data]     pages/index.xml'. A configured proxy performs the
[nltk_data]     egress, so NLTK cannot pin the validated IP and SSRF
[nltk_data]     protection cannot be enforced (CWE-918). If and only
[nltk_data]     if the proxy is trusted to be SSRF-safe, opt in via
[nltk_data]     NLTK_ALLOW_PROXIED_URLOPEN=1 or
[nltk_data]     nltk.pathsec.ALLOW_PROXIED_FETCH=True.


False

In [2]:
# Загрузка текста: каждая строка файла - одно предложение
with open("lab02_data.txt", encoding="utf-8") as f:
    sentences = [line.strip() for line in f if line.strip()]

print("Предложений:", len(sentences))

Предложений: 10


In [9]:
# Токенизация, удаление стоп-слов и стемминг
tokenizer = RegexpTokenizer(r"\w+")
stop = set(stopwords.words("english"))
stemmer = SnowballStemmer("english")
tokens = []
for s in sentences:
    words = [w.lower() for w in tokenizer.tokenize(s)]
    words = [w for w in words if w not in stop]
    tokens.append([stemmer.stem(w) for w in words])

print("Токены первого предложения:", tokens[0])

Токены первого предложения: ['roman', 'empir', 'expand', 'rapid', 'biggest', 'empir', 'world', 'long', 'time']


In [4]:
dict_tokens = corpora.Dictionary(tokens)
doc_term_mat = [dict_tokens.doc2bow(token) for token in tokens]

print("Размер словаря:", len(dict_tokens))

Размер словаря: 78


In [5]:
num_topics = 2
ldamodel = models.ldamodel.LdaModel(doc_term_mat, num_topics=num_topics,
                                    id2word=dict_tokens, passes=25, random_state=42)

In [6]:
# По пять представительных слов для каждой темы с их относительными вкладами
for i in range(num_topics):
    print(f"Тема {i}:")
    for word, weight in ldamodel.show_topic(i, topn=5):
        print(f"   {word:<15}{weight:.4f}")
    print()

Тема 0:
   empir          0.0352
   formul         0.0321
   time           0.0321
   histor         0.0209
   peopl          0.0209

Тема 1:
   mathemat       0.0379
   set            0.0271
   structur       0.0271
   call           0.0271
   later          0.0162



In [7]:
# Распределение тем по предложениям (для анализа)
for i, bow in enumerate(doc_term_mat):
    topics = ldamodel.get_document_topics(bow)
    best = max(topics, key=lambda t: t[1])
    print(f"{i}: тема {best[0]} ({float(best[1]):.3f}) | {sentences[i][:60]}")

0: тема 0 (0.947) | The Roman empire expanded very rapidly and it was the bigges
1: тема 1 (0.952) | An algebraic structure is a set with one or more finitary op
2: тема 1 (0.957) | Renaissance started as a cultural movement in Italy in the L
3: тема 0 (0.951) | The line of demarcation between prehistoric and historical t
4: тема 0 (0.935) | Mathematicians seek out patterns and use them to formulate n
5: тема 1 (0.935) | A notational symbol that represents a number is called a num
6: тема 1 (0.942) | The process of extracting the underlying essence of a mathem
7: тема 1 (0.849) | Historically, people have frequently waged wars against each
8: тема 0 (0.959) | Ancient history indicates that various outside influences ha
9: тема 1 (0.948) | Mappings between sets which preserve structures are of speci
